# Six Practical Machine Learning Training-Efficiency Tips

Complete companion on all 60,000/10,000 original MNIST images; both LLE examples use the original first 5,000 training rows. Python 3.11+, CPU, no keys/GPU; initial 11MB dataset download. Run top to bottom; full SVM training can take several minutes or longer.

MNIST: Yann LeCun and Corinna Cortes, derived from NIST; CC BY-SA 3.0. Official Keras-hosted NPZ, SHA-256 checked, allow_pickle=False. Direct NumPy loading returns exactly the same data without a TensorFlow dependency.

Maintenance: imports and stray syntax repaired; deterministic seeds and explicit initialization; the original hinge dual fit failed to converge at 20,000 iterations on full MNIST. The maintained LinearSVC uses the current squared-hinge primal solver (dual=False) for this n_samples > n_features case, with explicit tol=1e-2 to trade precision for runtime after the strict-tolerance primal run remained unfinished for over ten minutes; the original hinge version remains unchanged in the appendix. BLAS is one thread throughout. Parallel bagging uses all joblib logical CPUs via shared-memory threads to avoid process copies. Dense LLE eigensolver avoids ARPACK singular-factorization failures at the cost of different timing. Source timing scopes are retained (training blocks include training prediction; LLE includes plotting). No peak-memory benchmark or universal speed/accuracy guarantee is claimed. The arrays are still resident in these teaching examples; true out-of-core loading is a separate concern. Source prose is preserved on the website and substantive corrections are recorded in its import review.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==2.3.3', 'scipy==1.16.2', 'scikit-learn==1.7.2',
                'matplotlib==3.10.9', 'joblib==1.5.2', 'threadpoolctl==3.6.0',
                'certifi==2026.10.23'], check=True)
import hashlib
import os
import ssl
import tempfile
import time
import urllib.request
from pathlib import Path
import certifi
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pyplot
import sklearn
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from joblib import cpu_count, parallel_backend
from threadpoolctl import threadpool_limits
%matplotlib inline
thread_limits = threadpool_limits(limits=1)
np.set_printoptions(precision=8, linewidth=110)
timings = {}
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('Joblib logical CPUs:', cpu_count(), '| BLAS threads: 1')

In [ ]:
# Same NPZ and checksum used by keras.datasets.mnist.load_data; no TensorFlow runtime needed.
dataset_sha256 = '731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1'
dataset_path = Path(os.environ.get('TDB_MNIST_NPZ', str(Path(tempfile.gettempdir()) / 'tdb-mnist-731c5ac6.npz')))
if not dataset_path.exists():
    context = ssl.create_default_context(cafile=certifi.where())
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    with urllib.request.urlopen(url, context=context) as response, dataset_path.open('wb') as target:
        import shutil
        shutil.copyfileobj(response, target)
assert hashlib.sha256(dataset_path.read_bytes()).hexdigest() == dataset_sha256
with np.load(dataset_path, allow_pickle=False) as data:
    train_X, train_y = data['x_train'], data['y_train']
    test_X, test_y = data['x_test'], data['y_test']
pyplot.imshow(train_X[0], cmap=pyplot.get_cmap('gray'))
pyplot.axis('off')
pyplot.show()
print('MNIST SHA-256:', dataset_sha256)
print('First training label:', int(train_y[0]))

In [ ]:
print('The shape of the train data',train_X.shape)

print('The shape of the test data',test_X.shape)

In [ ]:
train_X_reshaped = train_X.reshape(train_X.shape[0], 28*28)
print('The shape of the reshaped train data', train_X_reshaped.shape)

test_X_reshaped = test_X.reshape(test_X.shape[0], 28*28)
print('The shape of the reshaped test data', test_X_reshaped.shape)
print('Training pixel-array payload MiB:', round(train_X_reshaped.nbytes / 2**20, 3))

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier


# get the start time
st = time.perf_counter()

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), 
    n_estimators=500,
    max_samples=100, 
    bootstrap=True, random_state=42)

bag_clf.fit(train_X_reshaped, train_y)

y_pred = bag_clf.predict(test_X_reshaped)

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
timings['bagging-serial'] = elapsed_time
bag_serial_predictions = y_pred.copy()
print('Test accuracy:', round(accuracy_score(test_y, y_pred), 6))

In [ ]:
# get the start time
st = time.perf_counter()


bag_clf_all_cores = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), 
    n_estimators=500,
    max_samples=100, 
    bootstrap=True,
    n_jobs=-1, random_state=42)

with parallel_backend('threading'):
    bag_clf_all_cores.fit(train_X_reshaped, train_y)
    y_pred = bag_clf_all_cores.predict(test_X_reshaped)

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
timings['bagging-parallel'] = elapsed_time
bag_parallel_predictions = y_pred.copy()
print('Test accuracy:', round(accuracy_score(test_y, y_pred), 6))

In [ ]:
# Calculate the execution time for the PCA using full SVD solver

# get the start time
st = time.perf_counter()

full_pca = PCA(n_components=100, svd_solver="full")
X_reduced_full = full_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time using full SVD', elapsed_time, 'seconds')
timings['pca-full'] = elapsed_time
print('Reduced shape:', X_reduced_full.shape)
print('Explained variance ratio:', round(full_pca.explained_variance_ratio_.sum(), 8))
print('Reduced array payload MiB:', round(X_reduced_full.nbytes/2**20, 3))

In [ ]:
# Calculate the execution time for the PCA using randomized solver

# get the start time
st = time.perf_counter()

rnd_pca = PCA(n_components=100, svd_solver="randomized", random_state=42)
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time using randomized solver', elapsed_time, 'seconds')
randomized_pca = rnd_pca
timings['pca-randomized'] = elapsed_time
print('Reduced shape:', X_reduced_rand.shape)
print('Explained variance ratio:', round(rnd_pca.explained_variance_ratio_.sum(), 8))

In [ ]:
from sklearn.decomposition import IncrementalPCA
n_batches = 100

# Calculate the execution time for the PCA using randomized solver

# get the start time
st = time.perf_counter()


inc_pca = IncrementalPCA(n_components=100)
for X_batch in np.array_split(train_X_reshaped, n_batches):
    inc_pca.partial_fit(X_batch)
    
X_reduced = inc_pca.transform(train_X_reshaped)

#get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time using incremental PCA', elapsed_time, 'seconds')
timings['pca-incremental'] = elapsed_time
print('Incremental samples seen:', int(inc_pca.n_samples_seen_))
print('Reduced shape:', X_reduced.shape)
print('Explained variance ratio:', round(inc_pca.explained_variance_ratio_.sum(), 8))

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, eigen_solver="dense")
X_reduced_LLE = lle.fit_transform(train_X_reshaped[0:5000,:])

plt.scatter(X_reduced_LLE[:, 0],X_reduced_LLE[:,1], c=train_y[0:5000], cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title("MNIST with LLE ")
plt.show()

#get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time using LLE', elapsed_time, 'seconds')
lle_direct = lle
embedding_direct = X_reduced_LLE.copy()
timings['lle-direct'] = elapsed_time
print('Embedding shape:', X_reduced_LLE.shape)
print('Reconstruction error:', format(lle.reconstruction_error_, '.10g'))

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()

rnd_pca = PCA(n_components=100, svd_solver="randomized", random_state=42)
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped[0:5000,:])

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, eigen_solver="dense")
X_reduced_LLE = lle.fit_transform(X_reduced_rand)

plt.scatter(X_reduced_LLE[:, 0],X_reduced_LLE[:,1], c=train_y[0:5000], cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title("MNIST with PCA & LLE ")
plt.show()

#get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Execution time using PCA + LLE', elapsed_time, 'seconds')
lle_chained = lle
embedding_chained = X_reduced_LLE.copy()
timings['lle-chained'] = elapsed_time
print('Embedding shape:', X_reduced_LLE.shape)
print('Reconstruction error:', format(lle.reconstruction_error_, '.10g'))
chain_pca = rnd_pca
chain_reduced = X_reduced_rand.copy()

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()

svm_clf = Pipeline([
("scaler", StandardScaler()),
("linear_svc", LinearSVC(C=1, loss="squared_hinge", dual=False, random_state=42, tol=1e-2, max_iter=10000)),
])

svm_clf.fit(train_X_reshaped, train_y)

y_train_predict = svm_clf.predict(train_X_reshaped)
print('Train accuarcy of linear SVC', accuracy_score(train_y, y_train_predict, normalize=True))


#get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Training time using Linear SVC', elapsed_time, 'seconds')
linear_train_predictions = y_train_predict.copy()
timings['linear-training-and-train-predict'] = elapsed_time
print('Maximum solver iterations:', int(np.max(svm_clf[-1].n_iter_)))

In [ ]:
# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()

y_test_predict = svm_clf.predict(test_X_reshaped)
print('Test accuarcy of linear SVC', accuracy_score(test_y, y_test_predict, normalize=True))

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Inference time using Linear SVC', elapsed_time, 'seconds')
linear_test_predictions = y_test_predict.copy()
timings['linear-test-predict'] = elapsed_time

In [ ]:
from sklearn.svm import SVC

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()


poly_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5, cache_size=512))
    ])
poly_kernel_svm_clf.fit(train_X_reshaped, train_y)

y_train_predict = poly_kernel_svm_clf.predict(train_X_reshaped)
print('Train accuarcy of  SVC', accuracy_score(train_y, y_train_predict, normalize=True))

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Training time using Kerenl SVC', elapsed_time, 'seconds')
kernel_train_predictions = y_train_predict.copy()
timings['kernel-training-and-train-predict'] = elapsed_time
print('Kernel support vectors:', int(poly_kernel_svm_clf[-1].n_support_.sum()))

In [ ]:
from sklearn.metrics import accuracy_score

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.perf_counter()

y_test_predict = poly_kernel_svm_clf.predict(test_X_reshaped)
print('Test accuarcy of  SVC', accuracy_score(test_y, y_test_predict, normalize=True))

# get the end time
et = time.perf_counter()

# get the execution time
elapsed_time = et - st
print('Inference time using Kerenl SVC', elapsed_time, 'seconds')
kernel_test_predictions = y_test_predict.copy()
timings['kernel-test-predict'] = elapsed_time

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# get the start time
st = time.perf_counter()

minibatch_kmeans = MiniBatchKMeans(n_clusters=9, n_init=10, batch_size=1024, random_state=42)
minibatch_kmeans.fit(train_X_reshaped)

# get the end time
et = time.perf_counter()


# get the execution time
elapsed_time = et - st
print('Computaional time using MiniBatch K-Means', elapsed_time, 'seconds')
timings['minibatch-kmeans'] = elapsed_time
print('MiniBatch inertia:', round(minibatch_kmeans.inertia_, 6))
print('Cluster sizes:', np.bincount(minibatch_kmeans.labels_, minlength=9).tolist())

In [ ]:
from sklearn.cluster import KMeans

# get the start time
st = time.perf_counter()

kmeans = KMeans(n_clusters=9, n_init=10, random_state=42)
kmeans.fit(train_X_reshaped)

# get the end time
et = time.perf_counter()


# get the execution time
elapsed_time = et - st
print('Computaional time using K-Means', elapsed_time, 'seconds')
timings['kmeans'] = elapsed_time
print('K-Means inertia:', round(kmeans.inertia_, 6))
print('Cluster sizes:', np.bincount(kmeans.labels_, minlength=9).tolist())

In [ ]:
from numpy.testing import assert_allclose, assert_array_equal
from sklearn.neighbors import NearestNeighbors
assert train_X.shape == (60000,28,28) and test_X.shape == (10000,28,28)
assert train_X_reshaped.shape == (60000,784) and test_X_reshaped.shape == (10000,784)
assert train_X.dtype == np.uint8 and train_y[0] == 5
assert_array_equal(bag_serial_predictions, bag_parallel_predictions)
for a,b in zip(bag_clf.estimators_samples_,bag_clf_all_cores.estimators_samples_):
    assert len(a)==100;assert_array_equal(a,b)
# Independent probability averaging on 128 test rows, not estimator ensemble.predict.
probs=np.zeros((128,10))
for tree, features in zip(bag_clf.estimators_,bag_clf.estimators_features_):
    values=tree.predict_proba(test_X_reshaped[:128,features])
    probs[:,tree.classes_.astype(int)] += values/500
assert_array_equal(np.argmax(probs,axis=1),bag_serial_predictions[:128])
for model in [full_pca,randomized_pca,inc_pca]:
    assert_allclose(model.components_ @ model.components_.T, np.eye(100), atol=1e-10)
    block=train_X_reshaped[:256].astype(float)
    manual=(block-model.mean_) @ model.components_.T
    assert_allclose(manual,model.transform(block),atol=1e-9)
assert inc_pca.n_samples_seen_==60000
for embedding, model, inputs in [(embedding_direct,lle_direct,train_X_reshaped[:5000]),(embedding_chained,lle_chained,chain_reduced)]:
    assert embedding.shape==(5000,2) and np.isfinite(embedding).all()
    assert_allclose(embedding.mean(axis=0),0,atol=1e-10)
    assert_allclose(embedding.T @ embedding,np.eye(2),atol=1e-8)
    points=np.asarray(inputs,dtype=float)
    neighbors=NearestNeighbors(n_neighbors=10,n_jobs=1).fit(points).kneighbors(return_distance=False)
    reconstructed=np.zeros_like(embedding)
    for i, near in enumerate(neighbors):
        differences=points[near]-points[i]
        gram=differences @ differences.T
        ridge=0.001*np.trace(gram) if np.trace(gram)>0 else 0.001
        gram.flat[::11] += ridge
        weights=np.linalg.solve(gram,np.ones(10));weights /= weights.sum()
        reconstructed[i]=weights @ embedding[near]
    residual=embedding-reconstructed
    assert_allclose(np.square(residual).sum(),model.reconstruction_error_,rtol=1e-4,atol=1e-9)
# Linear multiclass decision from standardized data and coefficients on the full test set.
scaled=svm_clf[0].transform(test_X_reshaped)
scores=scaled @ svm_clf[-1].coef_.T + svm_clf[-1].intercept_
assert_array_equal(svm_clf[-1].classes_[scores.argmax(axis=1)],linear_test_predictions)
assert svm_clf[-1].n_iter_ < svm_clf[-1].max_iter
# Independently verify the SVC's pairwise polynomial decision on a bounded 32-row probe.
svc=poly_kernel_svm_clf[-1]
probe=poly_kernel_svm_clf[0].transform(test_X_reshaped[:32])
kernel=(svc._gamma * (probe @ svc.support_vectors_.T)+svc.coef0)**svc.degree
offset=np.r_[0,np.cumsum(svc.n_support_)]
pair_scores=[];votes=np.zeros((len(probe),10),dtype=int)
for i in range(10):
    for j in range(i+1,10):
        si,sj=slice(offset[i],offset[i+1]),slice(offset[j],offset[j+1])
        score=kernel[:,si] @ svc.dual_coef_[j-1,si]+kernel[:,sj] @ svc.dual_coef_[i,sj]+svc.intercept_[len(pair_scores)]
        pair_scores.append(score)
        votes[:,i] += score>0;votes[:,j] += score<=0
old_shape=svc.decision_function_shape;svc.decision_function_shape='ovo'
assert_allclose(np.array(pair_scores).T,svc.decision_function(probe),rtol=1e-7,atol=1e-6)
svc.decision_function_shape=old_shape
assert_array_equal(votes.argmax(axis=1),kernel_test_predictions[:32])
# Exact nearest-centroid labels/inertia without sklearn's distance helper, in bounded blocks.
for model in [minibatch_kmeans,kmeans]:
    inertia=0.0
    for start in range(0,60000,1000):
        block=train_X_reshaped[start:start+1000].astype(float)
        d2=np.square(block[:,None,:]-model.cluster_centers_[None,:,:]).sum(axis=2)
        labels=d2.argmin(axis=1)
        assert_array_equal(labels,model.labels_[start:start+len(block)])
        inertia += d2[np.arange(len(block)),labels].sum()
    assert_allclose(inertia,model.inertia_,rtol=1e-10)
print('All sixteen recovered examples passed on the original 60000/10000 MNIST split.')
print('Serial/parallel bagging predictions and all 500 sampled training-index arrays agree.')
print('PCA transforms/orthogonality, both 5000-row LLE reconstruction objectives, full linear test scores and 45 kernel decision pairs pass.')
print('All 120000 nearest-centroid labels and both full-data inertias match independent calculations.')
print('Array payloads are not peak-process memory measurements; timings are this run, not universal speedups.')
thread_limits.restore_original_limits()

## Complete original gist snapshots (not executed)

### Original 1: 9ae8180e294e23054067c86b39c62404
Revision: `eaad52234384cdfd240e82b6e8d34ae295b149b3`

```python
# Loading and Plotting random image of the mnsit dataset
(train_X, train_y), (test_X, test_y) = mnist.load_data()

# plotting the first image of the MNSIt dataset
pyplot.imshow(train_X[0], cmap=pyplot.get_cmap('gray'))
pyplot.show()
```

### Original 2: 73b307effb943889f625e92a2b3b5c25
Revision: `c86f440db1b5d413793dd9908f38f66b1ff325ec`

```python
print('The shape of the train data',train_X.shape)

print('The shape of the test data',test_X.shape)
```

### Original 3: 5099119fd67e63946b63924add799ceb
Revision: `4c7062b155b1237360c7439c197c61b6b69292f8`

```python
train_X_reshaped = train_X.reshape(train_X.shape[0], 28*28)
print('The shape of the reshaped train data', train_X_reshaped.shape)

test_X_reshaped = test_X.reshape(test_X.shape[0], 28*28)
print('The shape of the reshaped test data', test_X_reshaped.shape)
```

### Original 4: be82808bd516c1ec72324633e65397cb
Revision: `78289c0b3543088f1dcfcf34b6e5f5ab18dac9fd`

```python
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier


# get the start time
st = time.time()

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), 
    n_estimators=500,
    max_samples=100, 
    bootstrap=True)

bag_clf.fit(train_X_reshaped, train_y)

y_pred = bag_clf.predict(test_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
```

### Original 5: be348475befe6eb8781d7395ee1957fa
Revision: `6792faea2d1392928ea5244585d849a6b575ea3f`

```python
# get the start time
st = time.time()


bag_clf_all_cores = BaggingClassifier(
    DecisionTreeClassifier(), 
    n_estimators=500,
    max_samples=100, 
    bootstrap=True,
    n_jobs = -1)

bag_clf_all_cores.fit(train_X_reshaped, train_y)

y_pred = bag_clf_all_cores.predict(test_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time:', elapsed_time, 'seconds')
```

### Original 6: d4c1447911b8d3b75dfba6e931e07b4b
Revision: `1e6ccb2a1c900c3af39caef9ee4e5042991189b5`

```python
# Calculate the execution time for the PCA using full SVD solver

# get the start time
st = time.time()

rnd_pca = PCA(n_components=100, svd_solver="full")
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using full SVD', elapsed_time, 'seconds')
```

### Original 7: 0d539fc317c90554762cafe646db22fb
Revision: `039d9b026ba679a6ce01ae7f2aa796114a9c3e00`

```python
# Calculate the execution time for the PCA using randomized solver

# get the start time
st = time.time()

rnd_pca = PCA(n_components=100, svd_solver="randomized")
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped)

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using randomized solver', elapsed_time, 'seconds')
```

### Original 8: 1f06cb268661a001d1299075ae884d4b
Revision: `0686f4ad8abda5e78521f410af5f32a98352862d`

```python
from sklearn.decomposition import IncrementalPCA
n_batches = 100

# Calculate the execution time for the PCA using randomized solver

# get the start time
st = time.time()


inc_pca = IncrementalPCA(n_components=100)
for X_batch in np.array_split(train_X_reshaped, n_batches):
    inc_pca.partial_fit(X_batch)
    
X_reduced = inc_pca.transform(train_X_reshaped)

#get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using incremental PCA', elapsed_time, 'seconds')
```

### Original 9: cc8aceb8b4f57f25d2a641d9439cd7b2
Revision: `d8cfb409cbb5b4394486b2b948f9887ce56d8128`

```python
from sklearn.manifold import LocallyLinearEmbedding

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced_LLE = lle.fit_transform(train_X_reshaped[0:5000,:])

plt.scatter(X_reduced_LLE[:, 0],X_reduced_LLE[:,1], c=train_y[0:5000], cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title(" Reduced swiss roll with LLE ")
plt.show()

#get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using incremental PCA', elapsed_time, 'seconds')
```

### Original 10: 9d8dd50e4f7724695bc7f700acb3a7f8
Revision: `c9f72c1abcb9d199888630e1c77c19d57f7edc0d`

```python
from sklearn.manifold import LocallyLinearEmbedding

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()

rnd_pca = PCA(n_components=100, svd_solver="randomized")
X_reduced_rand = rnd_pca.fit_transform(train_X_reshaped[0:5000,:])

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced_LLE = lle.fit_transform(X_reduced_rand)

plt.scatter(X_reduced_LLE[:, 0],X_reduced_LLE[:,1], c=train_y[0:5000], cmap=plt.cm.rainbow )
plt.xlabel("X") 
plt.ylabel("y")
plt.title(" Reduced swiss roll with PCA & LLE ")
plt.show()

#get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Execution time using incremental PCA', elapsed_time, 'seconds')
```

### Original 11: 3c01a3c715f158e687fd6e523d4c988d
Revision: `ed07c7b28cdf941007976f05a0aa29a62b7a9032`

```python
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()

svm_clf = Pipeline([
("scaler", StandardScaler()),
("linear_svc", LinearSVC(C=1, loss="hinge")),
])

svm_clf.fit(train_X_reshaped, train_y)

y_train_predict = svm_clf.predict(train_X_reshaped)
print('Train accuarcy of linear SVC', accuracy_score(train_y, y_train_predict, normalize=True))


#get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Training time using Linear SVC', elapsed_time, 'seconds')
```

### Original 12: 5ba60a51761227745bdcfaf5601feedc
Revision: `524a2abba97961ef8b057caab1f7226373b102ed`

```python
# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()

y_test_predict = svm_clf.predict(test_X_reshaped)
print('Test accuarcy of linear SVC', accuracy_score(test_y, y_test_predict, normalize=True))

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Inference time using Linear SVC', elapsed_time, 'seconds')c
```

### Original 13: 5f398d24fd3b622d4608440f0c853ed3
Revision: `2edbcbb2721cc0496b9cdfff5fc1a3d897063f39`

```python
from sklearn.svm import SVC

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()


poly_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5))
    ])
poly_kernel_svm_clf.fit(train_X_reshaped, train_y)

y_train_predict = poly_kernel_svm_clf.predict(train_X_reshaped)
print('Train accuarcy of  SVC', accuracy_score(train_y, y_train_predict, normalize=True))

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Training time using Kerenl SVC', elapsed_time, 'seconds')
```

### Original 14: de72e780d2834ec9381634cfea49af63
Revision: `993442d58db1f475288ff67dc60f3331747ae744`

```python
from sklearn.metrics import accuracy_score

# Calculate the execution time for the PCA using randomized solver
# get the start time
st = time.time()

y_test_predict = poly_kernel_svm_clf.predict(test_X_reshaped)
print('Test accuarcy of  SVC', accuracy_score(test_y, y_test_predict, normalize=True))

# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
print('Inference time using Kerenl SVC', elapsed_time, 'seconds')
```

### Original 15: fa19b185022c197de40ee1b2f84e2edb
Revision: `34b854cce37a83197bc83a634d6d3ab55fdfe837`

```python
from sklearn.cluster import MiniBatchKMeans

# get the start time
st = time.time()

minibatch_kmeans = MiniBatchKMeans(n_clusters=9)
minibatch_kmeans.fit(train_X_reshaped)

# get the end time
et = time.time()


# get the execution time
elapsed_time = et - st
print('Computaional time using MiniBatch K-Means', elapsed_time, 'seconds')
```

### Original 16: 70ab905e1a5dbea0c40d99632267d34a
Revision: `5b7c726fe36f1131e15d8ebe2ff595e8e1697ec8`

```python
from sklearn.cluster import KMeans

# get the start time
st = time.time()

kmeans = KMeans(n_clusters=9)
kmeans.fit(train_X_reshaped)

# get the end time
et = time.time()


# get the execution time
elapsed_time = et - st
print('Computaional time using K-Means', elapsed_time, 'seconds')
```

